# Production Baseline Window Analysis

Turkey hazelnut production has grown ~9,500 MT/year since 1960. The choice of baseline window for the production shortfall trigger affects which years fire and by how much.

This notebook compares:
- **Rolling windows**: 5yr, 7yr, 10yr (each dropping the single worst year)
- **Trend-adjusted**: shortfall relative to OLS linear trend fitted on full FAOSTAT history

Frost trigger years (april_dh ≥ 3.0) are overlaid throughout.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

ROOT = Path('..').resolve()
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
faostat = pd.read_csv(ROOT / 'data/raw/faostat/turkey_hazelnut_production.csv').set_index('year')
frost   = pd.read_csv(ROOT / 'data/raw/era5_frost_monthly.csv', index_col='year')

prod = faostat['production_mt']
frost_years = frost[frost['april_dh'] >= 3.0].index.tolist()

print(f'Production data: {prod.index.min()}–{prod.index.max()}  (n={len(prod)})')
print(f'Frost trigger years (april_dh ≥ 3.0): {sorted(frost_years)}')

In [ ]:
# ── Baseline helpers ───────────────────────────────────────────────────────────
def rolling_baseline(s: pd.Series, window: int, exclude_worst: int = 1) -> pd.Series:
    result = {}
    for yr in s.index:
        past = s.loc[(s.index >= yr - window) & (s.index < yr)]
        if len(past) >= window - exclude_worst:
            result[yr] = past.sort_values().iloc[exclude_worst:].mean()
        else:
            result[yr] = np.nan
    return pd.Series(result, name=f'baseline_{window}yr')

def trend_baseline(s: pd.Series) -> pd.Series:
    years = s.index.values.astype(float)
    coeffs = np.polyfit(years, s.values, 1)
    return pd.Series(np.polyval(coeffs, years), index=s.index, name='baseline_trend')

def shortfall(s: pd.Series, baseline: pd.Series) -> pd.Series:
    return ((s - baseline) / baseline).rename(baseline.name.replace('baseline_', 'shortfall_'))

# Compute baselines and shortfalls
windows = [5, 7, 10]
baselines = {w: rolling_baseline(prod, w) for w in windows}
baselines['trend'] = trend_baseline(prod)

shortfalls = {k: shortfall(prod, b) for k, b in baselines.items()}

# Trend slope for annotation
trend_coeffs = np.polyfit(prod.index.values.astype(float), prod.values, 1)
print(f'Trend slope: {trend_coeffs[0]/1000:.1f}k MT/year')
print(f'Implied production 1961: {np.polyval(trend_coeffs, 1961)/1000:.0f}k MT')
print(f'Implied production 2024: {np.polyval(trend_coeffs, 2024)/1000:.0f}k MT')

## 1. Raw Production + Baselines

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))

ax.bar(prod.index, prod / 1000, color='steelblue', alpha=0.4, width=0.8, label='Production (MT)')
ax.plot(baselines['trend'].index, baselines['trend'] / 1000, 'k--', lw=1.8, label='Linear trend')
ax.plot(baselines[5].index, baselines[5] / 1000, color='tomato', lw=1.4, label='5yr rolling baseline')
ax.plot(baselines[10].index, baselines[10] / 1000, color='darkorange', lw=1.4, ls='-.', label='10yr rolling baseline')

for yr in frost_years:
    if yr in prod.index:
        ax.axvline(yr, color='purple', alpha=0.35, lw=1.2)

ax.set_ylabel('000 MT')
ax.set_title('Turkey Hazelnut Production — FAOSTAT 1961–2024\n(purple lines = frost trigger years, april_dh ≥ 3.0)')
ax.legend(fontsize=9)
ax.set_xlim(1961, 2025)
plt.tight_layout()
plt.savefig(ROOT / 'notebooks/figures/production_raw.png', bbox_inches='tight')
plt.show()

## 2. Shortfall by Window — Full History

In [ ]:
colors = {5: 'tomato', 7: 'darkorange', 10: 'goldenrod', 'trend': 'black'}
labels = {5: '5yr rolling', 7: '7yr rolling', 10: '10yr rolling', 'trend': 'Trend-adjusted'}

fig, axes = plt.subplots(4, 1, figsize=(13, 12), sharex=True)

for ax, key in zip(axes, [5, 7, 10, 'trend']):
    sf = shortfalls[key].dropna()
    ax.bar(sf.index, sf * 100, color=[
        ('firebrick' if v < 0 else 'seagreen') for v in sf
    ], alpha=0.7, width=0.8)
    ax.axhline(0, color='black', lw=0.8)
    ax.axhline(-20, color='grey', lw=0.8, ls='--', alpha=0.6)
    ax.axhline(-35, color='grey', lw=0.8, ls=':', alpha=0.6)
    for yr in frost_years:
        if yr in sf.index:
            ax.axvline(yr, color='purple', alpha=0.4, lw=1.2)
    ax.set_ylabel('Shortfall (%)')
    ax.set_title(f'{labels[key]}  (drop worst 1 of window)', fontsize=10)
    ax.set_ylim(-65, 65)

axes[-1].set_xlabel('Year')
fig.suptitle('Production Shortfall vs Baseline — Sensitivity to Window Choice\n(grey dashes = -20% / -35% payout band boundaries; purple = frost years)', y=1.01)
plt.tight_layout()
plt.savefig(ROOT / 'notebooks/figures/production_shortfall_windows.png', bbox_inches='tight')
plt.show()

## 3. Shortfall Comparison Table — 1990–2024

In [ ]:
df = pd.DataFrame({'prod_mt': prod})
for key in [5, 7, 10, 'trend']:
    df[f'sf_{key}'] = (shortfalls[key] * 100).round(1)
df['april_dh'] = frost['april_dh']
df = df.loc[1990:2024]

# Highlight frost years
def highlight(row):
    if row.name in frost_years:
        return ['background-color: #e8d5f5'] * len(row)
    return [''] * len(row)

display_cols = ['prod_mt', 'april_dh', 'sf_5', 'sf_7', 'sf_10', 'sf_trend']
rename = {'prod_mt': 'Prod (MT)', 'april_dh': 'April DH',
          'sf_5': 'SF 5yr %', 'sf_7': 'SF 7yr %',
          'sf_10': 'SF 10yr %', 'sf_trend': 'SF trend %'}

df[display_cols].rename(columns=rename).style\
    .apply(highlight, axis=1)\
    .format({'Prod (MT)': '{:,.0f}', 'April DH': '{:.2f}',
             'SF 5yr %': '{:.1f}', 'SF 7yr %': '{:.1f}',
             'SF 10yr %': '{:.1f}', 'SF trend %': '{:.1f}'})

## 4. How Many Years Cross Each Payout Threshold?

In [ ]:
thresholds = [-20, -35, -50]
period = slice(1990, 2025)

rows = []
for key, label in labels.items():
    sf = shortfalls[key].loc[period] * 100
    n = len(sf.dropna())
    row = {'Baseline': label, 'n years': n}
    for t in thresholds:
        count = (sf < t).sum()
        row[f'< {t}%'] = f'{count}  ({count/n*100:.0f}%)'
    rows.append(row)

print('Trigger firing frequency 1990–2024:')
print(pd.DataFrame(rows).set_index('Baseline').to_string())

## 5. Scatter: Frost DH vs Production Shortfall

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4), sharey=False)

frost_1990 = frost.loc[1990:2024]

for ax, key, label in zip(axes, [5, 7, 10, 'trend'], labels.values()):
    sf = shortfalls[key].loc[1990:2024] * 100
    combined = pd.DataFrame({'sf': sf, 'april_dh': frost_1990['april_dh']}).dropna()

    frost_mask = combined['april_dh'] >= 3.0
    ax.scatter(combined.loc[~frost_mask, 'april_dh'],
               combined.loc[~frost_mask, 'sf'],
               color='steelblue', alpha=0.6, s=40, label='Quiet')
    ax.scatter(combined.loc[frost_mask, 'april_dh'],
               combined.loc[frost_mask, 'sf'],
               color='purple', alpha=0.8, s=60, zorder=5, label='Frost fired')

    for yr, row in combined[frost_mask].iterrows():
        ax.annotate(str(yr), (row['april_dh'], row['sf']),
                    textcoords='offset points', xytext=(4, 3), fontsize=7)

    ax.axhline(0, color='black', lw=0.7)
    ax.axhline(-20, color='grey', lw=0.7, ls='--')
    ax.set_xlabel('April DH')
    ax.set_ylabel('Production shortfall (%)' if ax == axes[0] else '')
    ax.set_title(label, fontsize=9)
    ax.legend(fontsize=8)

fig.suptitle('Frost DH vs Production Shortfall — does more frost = worse harvest?', y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'notebooks/figures/frost_vs_production.png', bbox_inches='tight')
plt.show()

## 6. Key Observations

Run this cell after reviewing the charts.

In [ ]:
print('=== WINDOW SENSITIVITY SUMMARY ===')
print()
print('Frost trigger years and shortfall under each method:')
frost_rows = df[df['april_dh'] >= 3.0][['prod_mt','april_dh','sf_5','sf_7','sf_10','sf_trend']]
print(frost_rows.rename(columns=rename).to_string())
print()
print('Large shortfalls with NO frost (april_dh < 1.0):')
no_frost_short = df[(df['april_dh'] < 1.0) & (df['sf_5'] < -15)][['prod_mt','april_dh','sf_5','sf_7','sf_10','sf_trend']]
print(no_frost_short.rename(columns=rename).sort_values('SF 5yr %').to_string())
print()
print('Window effect on trigger frequency (< -20% shortfall, 1990–2024):')
for key, label in labels.items():
    sf = shortfalls[key].loc[1990:2024] * 100
    n_fire = (sf < -20).sum()
    print(f'  {label:20s}: {n_fire}/{len(sf.dropna())} = {n_fire/len(sf.dropna())*100:.0f}%')